# RCM check 2 - Hosing response of the RCM and of EC-Earth3 over the PRUDENCE regions

**Ad hoc test, outside the chain**: it reads only the cache of monthly means written by
`1_hosing_signal_rcm_vs_gcm.ipynb` (run it first) and writes figures / tables in `outputs/` next to this notebook.

**Why.** Notebook 1 compared the hosing - no hosing changes of RegCM5 0.25 and EC-Earth3 (2050 pair) cell by cell:
the annual domain means agree, but over 20 years the change patterns are hardly reproducible within each model (noise
ceiling undefined for temperature in most seasons), so a cell-by-cell comparison cannot decide. Averaging over large
regions removes most of the internal variability. Question: over the 8 PRUDENCE regions, does the RCM reproduce the
change of its driver, in particular the stronger autumn cooling of the RCM over eastern Europe found in notebook 1?

**Method.**
- Same data, years, aggregation and domain as notebook 1: monthly means of 2125-2149 (cache), years `YEARS` (default
  2130-2149), RCM aggregated onto the EC-Earth3 grid outside the relaxation zone (cells covered at least `MIN_COVER`),
  LISFLOOD domain without the land outside Europe. Both models are therefore averaged over exactly the same GCM cells.
- PRUDENCE regions (Christensen and Christensen 2007, Climatic Change 81, 7-30): BI British Isles, IP Iberian
  Peninsula, FR France, ME Mid-Europe, SC Scandinavia, AL Alps, MD Mediterranean, EA Eastern Europe; a GCM cell
  belongs to a region if its centre lies in the box. Regional means weighted by cos(lat).
- Per region, variable (tasmax, tasmin in K; pr in % of the no-hosing mean) and season (DJF, MAM, JJA, SON, year):
  yearly seasonal means of the regional mean, change hosing - no hosing of each model, difference RCM - GCM; 90 %
  bootstrap intervals (`N_BOOT`, years of each of the four runs resampled independently).
- Agreement across regions: for each variable and season, the correlation and the regression slope of the 8 RCM
  changes on the 8 GCM changes.
- Context: RCM - GCM climate per region (no-hosing run): the difference that step 2 would remove.

**Reading.** For a region and season, the RCM change differs from that of its driver when the 90 % interval of
RCM - GCM excludes 0. With 8 regions x 5 seasons x 3 variables, about 6 of the 120 intervals are expected to exclude
0 by chance alone at the 90 % level: a disagreement is meaningful when it is coherent (several neighbouring regions,
or several variables, in the same season).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import sparse
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
RCM_TAGS = {"nohos": "2050_Base", "hos": "2050_Hose"}                     # as notebook 1
GCM_RUNS = {"nohos": "gcm_2050_nohosing", "hos": "gcm_2050_hosing"}
VARS = ["tasmax", "tasmin", "pr"]
DATA_YEARS = (2125, 2149)
YEARS = tuple(int(y) for y in os.environ.get("SIGNAL_YEARS", "2130,2149").split(","))
SPONGE = 11 + 2
MIN_COVER = 0.5
PR_MIN = 0.1
SEASONS = {"DJF": (12, 1, 2), "MAM": (3, 4, 5), "JJA": (6, 7, 8), "SON": (9, 10, 11), "year": tuple(range(1, 13))}
PRUDENCE = {"BI": ("British Isles", (50, 59), (-10, 2)), "IP": ("Iberian Peninsula", (36, 44), (-10, 3)),
            "FR": ("France", (44, 50), (-5, 5)), "ME": ("Mid-Europe", (48, 55), (2, 16)),
            "SC": ("Scandinavia", (55, 70), (5, 30)), "AL": ("Alps", (44, 48), (5, 15)),
            "MD": ("Mediterranean", (36, 44), (3, 25)), "EA": ("Eastern Europe", (44, 55), (16, 30))}   # lat, lon boxes
N_BOOT = 1000
SEED = 20261120

CACHE = bu.out_dir(cfg, "rcm_check", "cache")
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
TAG = f"{YEARS[0]}-{YEARS[1]}"
print(f"years {TAG} | cache of notebook 1: {CACHE}")

## 1. Monthly means from the cache of notebook 1, RCM aggregated onto the EC-Earth3 grid, regions

In [ ]:
def cached(path):
    assert path.exists(), f"missing {path.name}: run 1_hosing_signal_rcm_vs_gcm.ipynb first"
    return xr.open_dataarray(path).load()


RCM_NAT = {(run, v): cached(CACHE / f"rcm025_{RCM_TAGS[run]}_{v}_monthly_native_{DATA_YEARS[0]}-{DATA_YEARS[1]}.nc")
           for run in RCM_TAGS for v in VARS}
GCM = {(run, v): cached(CACHE / f"ecearth3_{GCM_RUNS[run]}_{v}_monthly_{DATA_YEARS[0]}-{DATA_YEARS[1]}.nc")
       for run in GCM_RUNS for v in VARS}
R0, G0 = RCM_NAT["nohos", "tasmax"], GCM["nohos", "tasmax"]
RLAT, RLON = R0["lat"].values.astype(float), ((R0["lon"].values.astype(float) + 180.0) % 360.0) - 180.0
GLAT, GLON = G0.lat.values, G0.lon.values
NY, NX = RLAT.shape; NG = GLAT.size * GLON.size
INNER = np.zeros((NY, NX), bool); INNER[SPONGE:NY - SPONGE, SPONGE:NX - SPONGE] = True

# aggregation onto the EC-Earth3 grid, as notebook 1
ye, xe = bu._edges(GLAT), bu._edges(GLON)
iy, ix = np.searchsorted(ye, RLAT) - 1, np.searchsorted(xe, RLON) - 1
ok = INNER & (iy >= 0) & (iy < GLAT.size) & (ix >= 0) & (ix < GLON.size)
COL = np.flatnonzero(ok); ROW = (iy * GLON.size + ix).ravel()[COL]
ONES = sparse.csr_matrix((np.ones(COL.size), (ROW, COL)), shape=(NG, NY * NX))


def cell_area_km2(lat, lon):
    R = 6371.0
    la, lo = np.deg2rad(lat), np.deg2rad(lon)
    d = lambda a1, o1, a2, o2: 2 * R * np.arcsin(np.sqrt(np.sin((a2 - a1) / 2) ** 2
                                                       + np.cos(a1) * np.cos(a2) * np.sin((o2 - o1) / 2) ** 2))
    dx = d(la[:, :-1], lo[:, :-1], la[:, 1:], lo[:, 1:]); dx = np.c_[dx, dx[:, -1:]]
    dy = d(la[:-1], lo[:-1], la[1:], lo[1:]); dy = np.r_[dy, dy[-1:]]
    return dx * dy


g_area = (np.diff(np.sin(np.deg2rad(np.clip(ye, -90, 90))))[:, None] * np.deg2rad(np.diff(xe))[None, :]) * 6371.0 ** 2
COVER = (np.bincount(ROW, weights=cell_area_km2(RLAT, RLON).ravel()[COL], minlength=NG) / g_area.ravel()).reshape(GLAT.size, GLON.size)


def to_gcm(a):
    flat = np.asarray(a, float).reshape(-1, NY * NX)
    fin = np.isfinite(flat)
    num = ONES @ np.where(fin, flat, 0.0).T
    den = ONES @ fin.T.astype(float)
    return np.where(den > 0, num / np.where(den > 0, den, 1), np.nan).T.reshape(flat.shape[0], GLAT.size, GLON.size)


RCM = {k: xr.DataArray(to_gcm(da.values), dims=("time", "lat", "lon"), coords={"time": da.time, "lat": GLAT, "lon": GLON})
       for k, da in RCM_NAT.items()}
DOM = bu.load_domain_mask(cfg, "ecearth3", like=G0.isel(time=0)).values & (COVER >= MIN_COVER)
LAT2 = np.broadcast_to(GLAT[:, None], DOM.shape); LON2 = np.broadcast_to(GLON[None, :], DOM.shape)
WREG = {}
for r, (name, la, lo) in PRUDENCE.items():
    m = DOM & (LAT2 >= la[0]) & (LAT2 <= la[1]) & (LON2 >= lo[0]) & (LON2 <= lo[1])
    w = np.where(m, np.cos(np.deg2rad(LAT2)), 0.0).ravel()
    WREG[r] = w / w.sum()
    print(f"{r} {name:18s}: {m.sum():3d} EC-Earth3 cells")

BORDERS = bu.borders(cfg, "10M")
fig, ax = plt.subplots(figsize=(8, 6))
ax.pcolormesh(GLON, GLAT, np.where(DOM, 1.0, np.nan), shading="nearest", cmap="Greys", vmin=0, vmax=2.5)
for r, (name, la, lo) in PRUDENCE.items():
    ax.add_patch(Rectangle((lo[0], la[0]), lo[1] - lo[0], la[1] - la[0], fill=False, ec="C3", lw=1.2))
    ax.text(np.mean(lo), np.mean(la), r, ha="center", va="center", color="C3", fontsize=10, weight="bold")
bu.draw_borders(ax, BORDERS, lw=0.3); ax.set_xlim(-15, 45); ax.set_ylim(33, 72)
ax.set_title("PRUDENCE regions on the domain of the comparison (grey: GCM cells used)")
plt.tight_layout(); plt.savefig(OUT / "prudence_regions.png", dpi=110); plt.show()

## 2. Regional changes, bootstrap intervals

In [ ]:
DAYS = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], float)


def season_means(da, months, years=YEARS):
    # yearly seasonal means (n_years, ...), months weighted by their days; DJF of year y = Dec y-1 + Jan, Feb y
    t = pd.DatetimeIndex(da["time"].values)
    djf = len(months) == 3 and 12 in months
    sy = t.year + (djf & (t.month == 12)).astype(int)
    v, out = da.values, []
    for y in range(years[0], years[1] + 1):
        k = np.flatnonzero(np.isin(t.month, months) & (sy == y))
        if len(k) == len(months):
            w = DAYS[t.month[k] - 1]
            out.append(np.tensordot(w / w.sum(), v[k], axes=1))
    return np.stack(out)


def regional(src, run, var, months):
    # yearly seasonal means of every regional mean: (n_years, n_regions)
    x = season_means(src[run, var], months).reshape(-1, DOM.size)
    return np.stack([np.nansum(x * WREG[r], axis=1) for r in PRUDENCE], axis=1)


def change(hos, nohos, var):
    return 100.0 * (hos / nohos - 1.0) if var == "pr" else hos - nohos


rng = np.random.default_rng(SEED)
rows, BOOT = [], {}
for var in VARS:
    for s, months in SEASONS.items():
        Y = {(m, run): regional(src, run, var, months) for m, src in (("GCM", GCM), ("RCM", RCM)) for run in ("nohos", "hos")}
        n = Y["GCM", "hos"].shape[0]
        dg = change(Y["GCM", "hos"].mean(0), Y["GCM", "nohos"].mean(0), var)
        dr = change(Y["RCM", "hos"].mean(0), Y["RCM", "nohos"].mean(0), var)
        bg, br = [], []
        for _ in range(N_BOOT):
            pick = {k: Y[k][rng.integers(0, n, n)].mean(0) for k in Y}
            bg.append(change(pick["GCM", "hos"], pick["GCM", "nohos"], var))
            br.append(change(pick["RCM", "hos"], pick["RCM", "nohos"], var))
        bg, br = np.array(bg), np.array(br)
        BOOT[var, s] = (dg, dr, bg, br)
        q = lambda a: np.quantile(a, [0.05, 0.95], axis=0)
        (g_lo, g_hi), (r_lo, r_hi), (d_lo, d_hi) = q(bg), q(br), q(br - bg)
        for i, r in enumerate(PRUDENCE):
            rows.append(dict(var=var, season=s, region=r, unit="%" if var == "pr" else "K",
                             gcm=dg[i], gcm_lo=g_lo[i], gcm_hi=g_hi[i], rcm=dr[i], rcm_lo=r_lo[i], rcm_hi=r_hi[i],
                             diff=dr[i] - dg[i], diff_lo=d_lo[i], diff_hi=d_hi[i],
                             different=bool(d_lo[i] > 0 or d_hi[i] < 0)))
    print(f"{var}: done")
REG = pd.DataFrame(rows)
REG.to_csv(OUT / f"prudence_changes_{TAG}.csv", index=False)
print(f"{int(REG['different'].sum())} of {len(REG)} intervals of RCM - GCM exclude 0 (about {0.1 * len(REG):.0f} expected by chance)")

## 3. Tables: changes per region and season, and where the RCM differs from its driver

In [ ]:
def fmt(r):
    u = "" if r.unit == "K" else "%"
    return f"{r.gcm:+.2f} / {r.rcm:+.2f}{u}" + (" *" if r.different else "")


for var in VARS:
    t = REG[REG["var"] == var].copy()
    t["GCM / RCM"] = [fmt(r) for r in t.itertuples()]
    print(f"\n{var}: change hosing - no hosing, EC-Earth3 / RegCM5 0.25 ({'%' if var == 'pr' else 'K'}); "
          f"* = 90 % interval of RCM - GCM excludes 0")
    display(t.pivot(index="region", columns="season", values="GCM / RCM").loc[list(PRUDENCE), list(SEASONS)])

DIFF = REG[REG["different"]][["var", "season", "region", "unit", "gcm", "rcm", "diff", "diff_lo", "diff_hi"]]
print("\nRegions and seasons where the RCM change differs from that of EC-Earth3:")
with pd.option_context("display.float_format", "{:.2f}".format, "display.width", 200):
    display(DIFF.sort_values(["var", "season", "region"]))

## 4. Agreement across regions, and figures

In [ ]:
rows = []
for var in VARS:
    for s in SEASONS:
        dg, dr, _, _ = BOOT[var, s]
        slope, icpt = np.polyfit(dg, dr, 1)
        rows.append(dict(var=var, season=s, r_regions=np.corrcoef(dg, dr)[0, 1], slope=slope,
                         mean_gcm=dg.mean(), mean_rcm=dr.mean(), n_different=int(REG[(REG["var"] == var) & (REG["season"] == s)]["different"].sum())))
ACR = pd.DataFrame(rows)
ACR.to_csv(OUT / f"prudence_agreement_{TAG}.csv", index=False)
with pd.option_context("display.float_format", "{:.2f}".format):
    display(ACR)

x = np.arange(len(PRUDENCE))
for var in VARS:
    unit = "%" if var == "pr" else "K"
    fig, axes = plt.subplots(1, len(SEASONS), figsize=(4.0 * len(SEASONS), 3.6), sharey=True)
    for ax, s in zip(axes, SEASONS):
        t = REG[(REG["var"] == var) & (REG["season"] == s)].set_index("region").loc[list(PRUDENCE)]
        ax.errorbar(x - 0.12, t["gcm"], yerr=[t["gcm"] - t["gcm_lo"], t["gcm_hi"] - t["gcm"]], fmt="o", capsize=2, label="EC-Earth3")
        ax.errorbar(x + 0.12, t["rcm"], yerr=[t["rcm"] - t["rcm_lo"], t["rcm_hi"] - t["rcm"]], fmt="s", capsize=2, label="RegCM5 0.25")
        for i, d in enumerate(t["different"]):
            if d:
                ax.axvspan(i - 0.4, i + 0.4, color="C3", alpha=0.12, lw=0)
        ax.axhline(0, c="k", lw=0.5); ax.set_xticks(x); ax.set_xticklabels(list(PRUDENCE), fontsize=8)
        ax.set_title(f"{var} {s}", fontsize=9)
    axes[0].set_ylabel(f"hosing - no hosing [{unit}]"); axes[0].legend(fontsize=7)
    fig.suptitle(f"{var}: regional changes, 2050 pair, {TAG} (shaded: 90 % interval of RCM - GCM excludes 0)", y=1.02)
    plt.tight_layout(); plt.savefig(OUT / f"prudence_{var}_{TAG}.png", dpi=110, bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, len(VARS), figsize=(5 * len(VARS), 4.4))
for ax, var in zip(axes, VARS):
    for k, s in enumerate(SEASONS):
        dg, dr, _, _ = BOOT[var, s]
        ax.scatter(dg, dr, label=s, s=22, color=f"C{k}")
    lim = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lim, lim, "k--", lw=0.7); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("EC-Earth3"); ax.set_ylabel("RegCM5 0.25"); ax.set_title(f"{var}: regional changes ({'%' if var == 'pr' else 'K'})", fontsize=9)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / f"prudence_scatter_{TAG}.png", dpi=110); plt.show()

## 5. Context: RCM - GCM climate per region (no-hosing run)

K for tasmax and tasmin, % of the GCM for pr (regional means, `YEARS`): the difference between the climates of the two
models that step 2 would remove.

In [ ]:
rows = []
for var in VARS:
    for s, months in SEASONS.items():
        for run in ("nohos", "hos"):
            g = regional(GCM, run, var, months).mean(0); r = regional(RCM, run, var, months).mean(0)
            d = 100.0 * (r / g - 1.0) if var == "pr" else r - g
            rows += [dict(var=var, season=s, run=run, region=k, value=d[i]) for i, k in enumerate(PRUDENCE)]
CLIM = pd.DataFrame(rows)
CLIM.to_csv(OUT / f"prudence_rcm_minus_gcm_{TAG}.csv", index=False)
for var in VARS:
    print(f"\n{var}, no hosing: RCM - GCM ({'%' if var == 'pr' else 'K'})")
    t = CLIM[(CLIM["var"] == var) & (CLIM["run"] == "nohos")]
    display(t.pivot(index="region", columns="season", values="value").loc[list(PRUDENCE), list(SEASONS)].round(2))